# Project 04 — Simple Linear Regression (dose-response)

**Scenario.** A dose-response experiment: a single predictor $x$ (the dose) drives a continuous response $y$ linearly, with Gaussian noise.

**New skill:** predictors, with priors on a **slope** and an **intercept**. **Key pitfall:** un-scaled predictors. The doses live far from zero ($\sim 100$ to $600$), which makes the raw intercept meaningless, priors hard to set, and the sampler's geometry poor. The fix is to **standardize** $x$.

In [ ]:
import sys, pathlib
sys.path.insert(0, r'/home/user/biofx_python/bayesian_workflow_portfolio')
sys.path.insert(0, str(pathlib.Path.cwd()))
import warnings; warnings.filterwarnings('ignore')

In [ ]:
import numpy as np
import pymc as pm
import arviz as az
import matplotlib.pyplot as plt
az.style.use('arviz-darkgrid')
RNG = 20240604

## Step 1 — Problem & data-generating story

The response is linear in dose with constant-variance Gaussian noise: $y_i = \alpha + \beta x_i + \varepsilon_i$, $\varepsilon_i\sim N(0,\sigma)$. **Assumptions made explicit:** (a) the relationship is linear, (b) noise is Gaussian with constant variance (homoscedastic), (c) doses are measured without error. We synthesize from known natural-scale $\alpha=2.0$, $\beta=0.015$, $\sigma=0.8$, with doses on $[100, 600]$.

In [ ]:
from data.generate_data import generate
data = generate()
x, y = data['x'], data['y']
print(f"n={data['n']}, doses in [{x.min():.0f}, {x.max():.0f}] (far from zero)")
print(f"standardization: x_mean={data['x_mean']:.1f}, x_sd={data['x_sd']:.1f}")
print('natural-scale truth:', data['truth_natural'])

In [ ]:
fig, ax = plt.subplots(figsize=(6,3.5))
ax.scatter(x, y, color='#4C72B0')
ax.set(xlabel='dose x (natural units)', ylabel='response y',
       title='dose-response — note x is far from zero')
plt.tight_layout()

## Step 2 — Model specification & the standardization fix

We fit on the **standardized** predictor $x_\text{std}=(x-\bar x)/s_x$:

$$y_i \sim \text{Normal}(\alpha + \beta\,x_{\text{std},i}, \sigma), \quad \alpha\sim N(0,5),\ \beta\sim N(0,5),\ \sigma\sim\text{HalfNormal}(2).$$

**Why standardize?** With raw doses on $[100,600]$: (1) the intercept $\alpha$ would be the response at $x=0$, a wild extrapolation far outside the data; (2) the slope is tiny ($\sim 0.015$) so a sensible prior width is hard to guess; (3) $\alpha$ and $\beta$ become strongly correlated, a banana-shaped posterior that NUTS samples poorly. Standardizing centers $x$ (so $\alpha$ is the response at the *mean* dose — interpretable) and scales it to unit SD (so $\beta$ is the change per 1 SD of dose, an $O(1)$ quantity). Now $N(0,5)$ priors are sensible for both, and the posterior geometry is clean.

In [ ]:
from model import build_model, fit, to_natural
model = build_model(data)   # standardize=True by default
model

## Step 3 — Prior predictive checks

We simulate dose-response lines implied by the prior on the standardized scale. $N(0,5)$ priors should imply a wide fan of plausible slopes and intercepts — not so wide they predict absurd responses, not so tight they forbid the true line.

In [ ]:
with model:
    prior = pm.sample_prior_predictive(draws=80, random_seed=RNG)
xs = np.linspace(data['x_std'].min(), data['x_std'].max(), 50)
a_pr = prior.prior['alpha'].values.ravel()
b_pr = prior.prior['beta'].values.ravel()
fig, ax = plt.subplots(figsize=(6,3.5))
for a_i, b_i in zip(a_pr[:60], b_pr[:60]):
    ax.plot(xs, a_i + b_i*xs, color='#55A868', alpha=0.25)
ax.set(xlabel='x_std', ylabel='implied response',
       title='prior predictive lines — wide but plausible')
plt.tight_layout()

## Step 4 — Inference (NUTS)

We sample with `draws=1000, tune=1000, chains=4`. Standardizing decorrelates $\alpha$ and $\beta$, so the posterior is well-conditioned and NUTS mixes efficiently — a direct, measurable payoff of the standardization fix.

In [ ]:
idata = fit(data, draws=1000, tune=1000, chains=4, seed=404)

## Step 5 — Computational diagnostics

Check $\hat R\approx 1.00$, ESS $\gtrsim 400$, and 0 divergences for $\alpha,\beta,\sigma$. With standardized $x$ the $\alpha$-$\beta$ correlation is near zero, so ESS is high. (On raw $x$ the correlation can exceed 0.99 and ESS collapses — the geometric cost of not standardizing.)

In [ ]:
print(az.summary(idata, var_names=['alpha', 'beta', 'sigma']))
print('divergences:', int(idata.sample_stats['diverging'].sum()))
print('standardized truth:', data['truth'])

In [ ]:
az.plot_trace(idata, var_names=['alpha', 'beta', 'sigma']); plt.tight_layout()

## Step 6 — Posterior predictive checks

We overlay the fitted regression line (with uncertainty) on the data, and run a standard PPC. A well-fit linear model has residuals with no structure and data that sit inside the posterior-predictive band across the dose range.

In [ ]:
az.plot_ppc(idata, num_pp_samples=100); plt.tight_layout()

In [ ]:
a = idata.posterior['alpha'].values.ravel()
b = idata.posterior['beta'].values.ravel()
xs = np.linspace(data['x_std'].min(), data['x_std'].max(), 50)
lines = a[:400,None] + b[:400,None]*xs[None,:]
lo, mid, hi = np.percentile(lines, [3, 50, 97], axis=0)
fig, ax = plt.subplots(figsize=(6,3.5))
ax.scatter(data['x_std'], y, color='#4C72B0', label='data')
ax.plot(xs, mid, 'k-', label='posterior mean line')
ax.fill_between(xs, lo, hi, color='gray', alpha=0.3, label='94% band')
ax.set(xlabel='x_std', ylabel='y', title='fitted line with uncertainty')
ax.legend(); plt.tight_layout()

## Step 7 — Model criticism & comparison

We criticize the fit two ways: (1) confirm the posterior brackets the known standardized truth, and (2) map the coefficients back to the **natural scale** and check they match the true $\alpha=2.0$, $\beta=0.015$. The back-mapping is the whole reason standardizing is safe: it is an exact, invertible reparameterization.

In [ ]:
nat = to_natural(idata, data)
print(f"recovered natural-scale: alpha={nat['alpha']:.3f}, beta={nat['beta']:.4f}")
print('true natural-scale:', data['truth_natural'])
print('the standardized fit maps back to the true natural-scale coefficients')

## Step 8 — Decision & communication

Report the dose effect on the interpretable natural scale: how much the response rises per unit dose, with uncertainty, and a predicted response at a dose of interest.

In [ ]:
b_post = idata.posterior['beta'].values.ravel()
beta_nat = b_post / data['x_sd']
lo, hi = np.percentile(beta_nat, [3, 97])
print(f'Dose effect (natural scale) = {beta_nat.mean():.4f} response/unit dose, '
      f'94% CI [{lo:.4f}, {hi:.4f}]')
dose_q = 400.0
a_post = idata.posterior['alpha'].values.ravel()
x_std_q = (dose_q - data['x_mean']) / data['x_sd']
pred = a_post + b_post * x_std_q
print(f'Predicted response at dose {dose_q:.0f}: {pred.mean():.3f} '
      f'(94% CI [{np.percentile(pred,3):.3f}, {np.percentile(pred,97):.3f}])')

**Conclusion (for a collaborator).** The response rises by about 0.015 units per unit of dose (94% CI roughly [0.011, 0.019]). At a dose of 400 the expected response is about 8. We standardized the dose internally for stable estimation, then reported everything back on the natural dose scale the collaborator uses.